In [1]:
# ============================================
# QUESTION 7: Vision Transformers in Keras
# COMPLETE SELF-CONTAINED COLAB NOTEBOOK
# ============================================

import os
import numpy as np
import random
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

print("="*60)
print("QUESTION 7: Vision Transformers in Keras")
print("="*60)

# ============================================
# STEP 1: Create sample dataset
# ============================================

print("\n📁 Creating sample dataset...")

def create_sample_dataset():
    os.makedirs('./images_dataSAT/class_0_non_agri/', exist_ok=True)
    os.makedirs('./images_dataSAT/class_1_agri/', exist_ok=True)

    for i in range(30):
        img = np.random.randint(0, 255, (84, 80, 3), dtype=np.uint8)
        if i % 2 == 0:
            img[20:60, 30:50] = [200, 200, 200]
        cv2.imwrite(f'./images_dataSAT/class_0_non_agri/non_agri_{i:03d}.png', img)

    for i in range(35):
        img = np.random.randint(0, 255, (84, 80, 3), dtype=np.uint8)
        for _ in range(5):
            x = random.randint(0, 70)
            y = random.randint(0, 74)
            img[y:y+10, x:x+10] = [50, 200, 50]
        cv2.imwrite(f'./images_dataSAT/class_1_agri/agri_{i:03d}.png', img)

    print("✅ Sample dataset created!")

if not os.path.exists('./images_dataSAT'):
    create_sample_dataset()
else:
    print("✅ Dataset already exists!")

# ============================================
# Task 1: Load and summarize pre-trained CNN model
# ============================================

print("\n" + "="*50)
print("Task 1: Load and summarize pre-trained CNN model")
print("="*50)

pretrained_model = tf.keras.applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(80, 84, 3)
)
pretrained_model.trainable = False

print("✅ ResNet50 loaded successfully!")
print(f"Total layers: {len(pretrained_model.layers)}")
pretrained_model.summary()

# ============================================
# Task 2: Identify feature extraction layer
# ============================================

print("\n" + "="*50)
print("Task 2: Identify feature extraction layer")
print("="*50)

feature_layer_name = 'conv5_block3_out'
feature_layer = pretrained_model.get_layer(feature_layer_name)
print(f"✅ Feature extraction layer: {feature_layer_name}")
print(f"Layer details: {feature_layer}")

# ============================================
# Task 3: Define hybrid model
# ============================================

print("\n" + "="*50)
print("Task 3: Define hybrid model")
print("="*50)

def build_cnn_vit_hybrid(input_shape=(80, 84, 3), num_classes=1):
    inputs = tf.keras.Input(shape=input_shape)
    cnn_features = pretrained_model(inputs)

    # Project to patches for ViT
    x = layers.Conv2D(64, (1, 1), activation='relu')(cnn_features)
    patches = layers.Reshape((-1, 64))(x)

    # Transformer blocks
    for _ in range(4):
        attn = layers.MultiHeadAttention(num_heads=4, key_dim=64)(patches, patches)
        x = layers.Add()([patches, attn])
        x = layers.LayerNormalization()(x)
        ffn = layers.Dense(64, activation='relu')(x)
        ffn = layers.Dense(64)(ffn)
        x = layers.Add()([x, ffn])
        patches = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(patches)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return tf.keras.Model(inputs, outputs)

hybrid_model = build_cnn_vit_hybrid()
print("✅ Hybrid model defined successfully!")

# ============================================
# Task 4: Compile the hybrid_model
# ============================================

print("\n" + "="*50)
print("Task 4: Compile the hybrid_model")
print("="*50)

hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print("✅ Model compiled successfully!")

# ============================================
# Task 5: Set

QUESTION 7: Vision Transformers in Keras

📁 Creating sample dataset...
✅ Sample dataset created!

Task 1: Load and summarize pre-trained CNN model
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
✅ ResNet50 loaded successfully!
Total layers: 175


Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 80, 84, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 86, 90, 3) │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 40, 42,    │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 40, 42,    │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 40, 42,    │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 42, 44,    │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 20, 21,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 20, 21,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 20, 21,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 20, 21,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 20, 21,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 20, 21,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 20, 21,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 20, 21,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 20, 21,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 20, 21,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 20, 21,    │      1,024 │ conv2_block1_3_c

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)


Task 2: Identify feature extraction layer
✅ Feature extraction layer: conv5_block3_out
Layer details: <Activation name=conv5_block3_out, built=True>

Task 3: Define hybrid model
✅ Hybrid model defined successfully!

Task 4: Compile the hybrid_model
✅ Model compiled successfully!
